# **Online preference optimisation for protein design**

### A Colab tutorial accompanying *Steering Generative Models for Protein Design: Aligning and Conditioning Strategies*

This practical turns one idea from the review by **Stocco, Garibbo and Ferruz** into a small, inspectable experiment: repeatedly sample proteins from an autoregressive protein language model, score them with an oracle, convert the scores into preferences, and update the model with Direct Preference Optimisation (DPO).

The default objective is deliberately simple—**generate longer sequences**—so the whole feedback loop can run on a standard Colab GPU. Optional sections replace length with **mean pLDDT** or connect the loop to **CLEAN enzyme-function predictions**.

> **Teaching scope.** This notebook demonstrates the mechanics of alignment. Generated sequences are computational hypotheses, not validated proteins, and must not be treated as functional, stable, safe or experimentally ready.

## **Learning objectives**

By the end of the tutorial, you should be able to:

1. distinguish **parameter-updating alignment** from parameter-fixed steering;
2. explain how a scalar score can be converted into chosen/rejected sequence pairs;
3. implement an **iterative (online) DPO-style loop** for an autoregressive protein language model;
4. exchange one scoring oracle for another without rewriting the optimiser; and
5. recognise reward hacking and the limits of pLDDT- or function-predictor-based optimisation.

<div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin:18px 0;font-size:16px">
  <span style="padding:10px 14px;border-radius:10px;background:#E8F3FF"><b>Current policy</b></span>
  <span>→</span>
  <span style="padding:10px 14px;border-radius:10px;background:#ECF8F0"><b>Sample sequences</b></span>
  <span>→</span>
  <span style="padding:10px 14px;border-radius:10px;background:#FFF3D9"><b>Oracle scores</b></span>
  <span>→</span>
  <span style="padding:10px 14px;border-radius:10px;background:#F5EAFE"><b>Preference pairs</b></span>
  <span>→</span>
  <span style="padding:10px 14px;border-radius:10px;background:#FFE9E7"><b>DPO update</b></span>
  <span>↺</span>
</div>

## **1. From steering to alignment**

A pretrained generative model approximates the distribution of its training sequences, $p(x)$. Protein design instead asks for sequences with a desired property $y$, conceptually shifting generation towards $p(x\mid y)$.

The review separates two broad strategies:

- **Parameter-fixed steering** changes the prompt, activations, decoding rule or sampling procedure without updating the model weights.
- **Parameter-updating alignment** changes model parameters using examples, rewards or preferences. SFT, policy-gradient methods, GRPO and DPO belong here.

In this notebook, the pretrained **ProtGPT3-112M** model is the initial policy. Only small LoRA adapter matrices are trained; the pretrained weights stay frozen and also serve as the fixed reference policy.

### **1.1 What does “online DPO” mean here?**

Ordinary DPO is often trained once on a fixed preference dataset. Here, preferences are regenerated from the current policy at every round:

1. sample sequences from $\pi_\theta$;
2. calculate a scalar oracle score $r(x)$;
3. rank the samples and pair a preferred sequence $x_w$ with a dispreferred sequence $x_l$;
4. take DPO gradient steps; and
5. sample again from the updated policy.

This makes the **overall data-collection loop iterative/on-policy**, although each update is still a DPO update. The oracle supplies rankings, not gradients through the biological property.

We use the reference-aware DPO objective:

$$
\mathcal{L}_{\mathrm{DPO}} =
-\mathbb{E}\left[
\log \sigma\left(
\beta\left[
\log\frac{\pi_\theta(x_w)}{\pi_{\mathrm{ref}}(x_w)}
-
\log\frac{\pi_\theta(x_l)}{\pi_{\mathrm{ref}}(x_l)}
\right]
\right)
\right].
$$

$\beta$ controls the strength of preference separation. The reference terms discourage the aligned policy from moving arbitrarily far from the pretrained distribution.

The accompanying **ProtRL** repository provides reusable Weighted DPO, GRPO and REINFORCE trainers. We implement a minimal pairwise DPO step explicitly here so that every probability term remains visible to students.

### **1.2 Three possible scoring oracles**

| Objective | Example reward | What it measures | Important limitation |
|---|---:|---|---|
| **Length** | number of residues | Whether the generated chain is longer | Trivial to game by avoiding the end token; says nothing about fold or function |
| **Mean pLDDT** | mean ESMFold pLDDT | A structure predictor's local confidence in its own prediction | Not experimental stability, activity or proof of a correct fold; predictors can be exploited |
| **CLEAN** | confidence/distance for a target EC number | Sequence-based evidence for an Enzyme Commission class | An annotation prediction, not demonstrated catalysis; full local inference is computationally heavy |
| **Composite** | weighted, rank-normalised scores | A negotiated trade-off between objectives | Weights encode design choices and can hide failure in one component |

We start with length because it is fast and its failure mode is easy to see. That failure is pedagogically useful: optimisation finds what the reward literally asks for, not what we intended.

## **2. Colab setup**

Select **Runtime → Change runtime type → T4 GPU** before running the notebook.

- The default length experiment is designed as a short classroom demonstration.
- The pLDDT extension loads ESMFold and is substantially slower and more memory-intensive. Use fewer samples and shorter sequences.
- CLEAN's official local workflow requires large model/data downloads, so this notebook provides an export/parser interface rather than silently installing several gigabytes.

In [ ]:
%pip install -q "transformers>=4.51,<5" "peft>=0.15,<1" "accelerate>=1.6,<2" sentencepiece protobuf biopython einops dm-tree seaborn

In [ ]:
import gc
import math
import random
import re
import time
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

sns.set_theme(style="whitegrid", context="notebook")

assert torch.cuda.is_available(), (
    "A GPU is required for this practical. In Colab select "
    "Runtime → Change runtime type → T4 GPU, then run again."
)

DEVICE = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

### **2.1 Experiment configuration**

The defaults favour a visible result over a production-quality alignment run. Useful reward modes are:

- `"length"` — one-click default;
- `"target_length"` — rewards proximity to a specified length and avoids unbounded growth;
- `"plddt"` — optional ESMFold oracle; and
- `"length_plddt"` — rank-normalised combination of length and pLDDT.

For pLDDT modes, first reduce `samples_per_round` and `evaluation_samples` to 6–8, `rounds` to 1–2 and `max_new_tokens` to about 96. A high-RAM runtime may be required when ESMFold is first loaded.

In [ ]:
@dataclass
class ExperimentConfig:
    model_id: str = "AI4PD/ProtGPT3-112M"
    prompt: str = "1"                 # ProtGPT3: 1 = N→C; 2 = C→N
    reward_mode: str = "length"       # length | target_length | plddt | length_plddt
    target_length: int = 100

    rounds: int = 3
    samples_per_round: int = 20
    evaluation_samples: int = 20
    generation_batch_size: int = 5
    max_new_tokens: int = 128
    min_new_tokens: int = 24
    temperature: float = 1.0
    top_p: float = 0.95

    pairs_per_round: int = 8
    dpo_epochs_per_round: int = 3
    train_batch_size: int = 2
    learning_rate: float = 2e-4
    beta: float = 0.10
    max_grad_norm: float = 1.0

    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    seed: int = 17


CFG = ExperimentConfig()
display(pd.Series(asdict(CFG), name="value").to_frame())

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)

## **3. Load the policy and attach LoRA adapters**

ProtGPT3 is a decoder-only, character-level protein language model. Its direction token `1` requests generation from the N terminus to the C terminus. The leading beginning-of-sequence token is also required; omitting either moves generation away from the format seen during pretraining.

LoRA freezes the original model and learns small low-rank updates in selected attention projections. This gives us two policies in one object:

- adapters **enabled**: the trainable policy $\pi_\theta$;
- adapters **disabled**: the fixed pretrained reference $\pi_{\mathrm{ref}}$.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CFG.model_id,
    add_bos_token=True,
    add_eos_token=False,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    CFG.model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False
base_model.to(DEVICE)

module_names = {name.rsplit(".", 1)[-1] for name, _ in base_model.named_modules()}
target_modules = [name for name in ("q_proj", "v_proj") if name in module_names]
assert target_modules, "Could not find q_proj/v_proj attention modules in this checkpoint."

lora_config = LoraConfig(
    r=CFG.lora_rank,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

policy = get_peft_model(base_model, lora_config)
policy.print_trainable_parameters()

## **4. Generate and inspect protein sequences**

Autoregressive generation samples one residue at a time from

$$p(x)=\prod_{i=1}^{N}p(x_i\mid x_{<i}).$$

Temperature and nucleus (`top_p`) sampling control diversity. They are **sampling controls**, not learned alignment: changing them does not update the model parameters.

In [ ]:
AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")


def clean_sequence(text):
    # Remove tokenizer spacing/special text and retain the 20 canonical residues.
    compact = re.sub(r"\s+", "", text.upper())
    return "".join(char for char in compact if char in AMINO_ACIDS)


def residue_entropy(sequence):
    if not sequence:
        return np.nan
    _, counts = np.unique(list(sequence), return_counts=True)
    probabilities = counts / counts.sum()
    return float(-(probabilities * np.log2(probabilities)).sum())


def max_residue_fraction(sequence):
    if not sequence:
        return np.nan
    _, counts = np.unique(list(sequence), return_counts=True)
    return float(counts.max() / counts.sum())


@torch.inference_mode()
def sample_sequences(model, n_sequences, seed, use_reference=False):
    # Generate raw amino-acid completions, excluding prompt and special tokens.
    seed_everything(seed)
    model.eval()
    prompt_batch = tokenizer(
        CFG.prompt,
        add_special_tokens=True,
        return_tensors="pt",
    ).to(DEVICE)
    prompt_length = prompt_batch["input_ids"].shape[1]
    generated_sequences = []

    adapter_context = model.disable_adapter() if use_reference else nullcontext()
    with adapter_context:
        for start in range(0, n_sequences, CFG.generation_batch_size):
            current_batch_size = min(CFG.generation_batch_size, n_sequences - start)
            input_ids = prompt_batch["input_ids"].repeat(current_batch_size, 1)
            attention_mask = prompt_batch["attention_mask"].repeat(current_batch_size, 1)

            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                do_sample=True,
                temperature=CFG.temperature,
                top_p=CFG.top_p,
                min_new_tokens=CFG.min_new_tokens,
                max_new_tokens=CFG.max_new_tokens,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )
            completion_ids = generated[:, prompt_length:]
            decoded = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)
            generated_sequences.extend(clean_sequence(text) for text in decoded)

    return generated_sequences


def describe_sequences(sequences):
    frame = pd.DataFrame({"sequence": sequences})
    frame["length"] = frame["sequence"].str.len()
    frame["entropy_bits"] = frame["sequence"].map(residue_entropy)
    frame["max_residue_fraction"] = frame["sequence"].map(max_residue_fraction)
    frame["valid"] = frame["sequence"].map(lambda x: bool(x) and set(x) <= AMINO_ACIDS)
    return frame

In [ ]:
# A first sample from the unaligned reference policy
baseline_preview = sample_sequences(
    policy,
    n_sequences=CFG.evaluation_samples,
    seed=CFG.seed + 1_000,
    use_reference=True,
)
baseline_preview_df = describe_sequences(baseline_preview)
display(baseline_preview_df.head(8))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(baseline_preview_df, x="length", bins=12, ax=axes[0], color="#2878B5")
axes[0].axvline(CFG.max_new_tokens, ls="--", color="black", lw=1, label="generation cap")
axes[0].legend()
sns.histplot(baseline_preview_df, x="entropy_bits", bins=12, ax=axes[1], color="#41AB5D")
axes[1].set_title("Residue-composition entropy")
plt.tight_layout()
plt.show()

## **5. Define the reward oracle**

For the length exercise, higher reward means more residues. The numerical scale does not affect pair construction; only ranking matters. The target-length alternative is often better posed because it rewards proximity to a design specification rather than indefinite growth.

In [ ]:
def length_scores(sequences):
    return np.asarray([len(sequence) for sequence in sequences], dtype=float)


def target_length_scores(sequences, target):
    # Highest value (0) at the target; increasingly negative away from it.
    return -np.asarray([abs(len(sequence) - target) for sequence in sequences], dtype=float)


def rank_to_unit_interval(values):
    # Convert incomparable oracle scales to within-batch percentile ranks.
    return pd.Series(values).rank(method="average", pct=True).to_numpy(dtype=float)

### **5.1 Optional oracle: mean pLDDT from ESMFold**

pLDDT is a **per-residue confidence score produced by a structure predictor**. Averaging it gives a convenient scalar oracle, but its interpretation is narrow: a high mean pLDDT says the model is confident in its predicted local structure. It does not establish that the protein is stable, soluble, functional or correctly assembled.

The function below is real rather than a mock score. It loads ESMFold only if a pLDDT reward mode is selected, scores one sequence at a time, and caches results. The first call downloads a large checkpoint and may take several minutes.

In [ ]:
_ESMFOLD = {"model": None, "tokenizer": None}
_PLDDT_CACHE = {}


def _load_esmfold():
    if _ESMFOLD["model"] is not None:
        return

    from transformers import EsmForProteinFolding

    print("Loading ESMFold. This is the slow, memory-intensive optional path...")
    fold_tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
    fold_model = EsmForProteinFolding.from_pretrained(
        "facebook/esmfold_v1",
        low_cpu_mem_usage=True,
    )
    # Keep the folding trunk in fp32; halve the large ESM language-model stem.
    fold_model.esm = fold_model.esm.half()
    fold_model.trunk.set_chunk_size(64)
    fold_model.eval().to(DEVICE)
    torch.backends.cuda.matmul.allow_tf32 = True

    _ESMFOLD.update(model=fold_model, tokenizer=fold_tokenizer)


@torch.inference_mode()
def mean_plddt_scores(sequences):
    _load_esmfold()
    fold_model = _ESMFOLD["model"]
    fold_tokenizer = _ESMFOLD["tokenizer"]

    scores = []
    for index, sequence in enumerate(sequences, start=1):
        if sequence not in _PLDDT_CACHE:
            inputs = fold_tokenizer(
                sequence,
                return_tensors="pt",
                add_special_tokens=False,
            ).to(DEVICE)
            outputs = fold_model(**inputs)
            residue_mask = inputs["attention_mask"].bool()
            mean_score = outputs.plddt[residue_mask].float().mean().item()
            _PLDDT_CACHE[sequence] = mean_score
            del inputs, outputs
            torch.cuda.empty_cache()
        scores.append(_PLDDT_CACHE[sequence])
        print(f"ESMFold {index:>2}/{len(sequences)}", end="\r")
    print()
    return np.asarray(scores, dtype=float)

In [ ]:
def score_sequences(sequences):
    mode = CFG.reward_mode.lower()

    if mode == "length":
        return length_scores(sequences)
    if mode == "target_length":
        return target_length_scores(sequences, CFG.target_length)
    if mode == "plddt":
        return mean_plddt_scores(sequences)
    if mode == "length_plddt":
        length_component = rank_to_unit_interval(length_scores(sequences))
        plddt_component = rank_to_unit_interval(mean_plddt_scores(sequences))
        return 0.25 * length_component + 0.75 * plddt_component

    raise ValueError(
        f"Unknown reward_mode={CFG.reward_mode!r}. Choose length, target_length, "
        "plddt or length_plddt."
    )

## **6. Turn scores into preferences**

DPO needs comparisons rather than absolute scores. Within each sampled batch, we match the highest-scoring sequences with the lowest-scoring sequences. A larger reward gap provides a clearer teaching signal, while duplicate sequences and tied pairs are excluded.

This is only one preference-construction strategy. Quantile pairing, noisy comparisons, Pareto dominance and experimentally measured pairwise outcomes are important alternatives.

In [ ]:
def make_preference_pairs(scored_frame, max_pairs):
    ranked = (
        scored_frame[["sequence", "reward"]]
        .drop_duplicates("sequence")
        .sort_values("reward")
        .reset_index(drop=True)
    )
    n_pairs = min(max_pairs, len(ranked) // 2)
    if n_pairs == 0:
        raise ValueError("Not enough distinct sequences to form a preference pair.")

    rejected = ranked.head(n_pairs).reset_index(drop=True)
    chosen = ranked.tail(n_pairs).sort_values("reward", ascending=False).reset_index(drop=True)
    pairs = pd.DataFrame({
        "chosen": chosen["sequence"],
        "chosen_reward": chosen["reward"],
        "rejected": rejected["sequence"],
        "rejected_reward": rejected["reward"],
    })
    pairs["reward_gap"] = pairs["chosen_reward"] - pairs["rejected_reward"]
    pairs = pairs[pairs["reward_gap"] > 0].reset_index(drop=True)

    if pairs.empty:
        raise ValueError(
            "All sampled rewards are tied. Increase sampling diversity or choose another oracle."
        )
    return pairs

In [ ]:
preview_scored = baseline_preview_df.copy()
preview_scored["reward"] = score_sequences(preview_scored["sequence"].tolist())
preview_pairs = make_preference_pairs(preview_scored, CFG.pairs_per_round)
display(preview_pairs[["chosen_reward", "rejected_reward", "reward_gap"]].head())

## **7. Implement the DPO update**

Each input consists of the ProtGPT3 prompt followed by an amino-acid completion and an end token. We sum log-probabilities **only over the completion**, not over the prompt or padding.

The reference policy is evaluated with the LoRA adapter disabled. Because the pretrained weights remain frozen, it stays fixed across all rounds without requiring a second full model in GPU memory.

In [ ]:
def encode_completions(sequences):
    prompt_ids = tokenizer(
        CFG.prompt,
        add_special_tokens=True,
    )["input_ids"]

    encoded = []
    completion_masks = []
    for sequence in sequences:
        completion_ids = tokenizer(
            sequence,
            add_special_tokens=False,
        )["input_ids"]
        completion_ids = completion_ids + [tokenizer.eos_token_id]
        encoded.append(prompt_ids + completion_ids)
        completion_masks.append([0] * len(prompt_ids) + [1] * len(completion_ids))

    max_length = max(map(len, encoded))
    padded_ids, attention_masks, padded_completion_masks = [], [], []
    for token_ids, completion_mask in zip(encoded, completion_masks):
        padding = max_length - len(token_ids)
        padded_ids.append(token_ids + [tokenizer.pad_token_id] * padding)
        attention_masks.append([1] * len(token_ids) + [0] * padding)
        padded_completion_masks.append(completion_mask + [0] * padding)

    return {
        "input_ids": torch.tensor(padded_ids, dtype=torch.long, device=DEVICE),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long, device=DEVICE),
        "completion_mask": torch.tensor(
            padded_completion_masks, dtype=torch.float32, device=DEVICE
        ),
    }


def completion_log_probabilities(model, batch):
    logits = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        use_cache=False,
    ).logits.float()

    # A causal model's logit at position t predicts the token at t + 1.
    shifted_logits = logits[:, :-1, :]
    shifted_token_ids = batch["input_ids"][:, 1:]
    shifted_completion_mask = batch["completion_mask"][:, 1:]

    token_logps = F.log_softmax(shifted_logits, dim=-1).gather(
        dim=-1,
        index=shifted_token_ids.unsqueeze(-1),
    ).squeeze(-1)
    return (token_logps * shifted_completion_mask).sum(dim=-1)


@torch.no_grad()
def reference_log_probabilities(sequences):
    was_training = policy.training
    policy.eval()
    batch = encode_completions(sequences)
    with policy.disable_adapter():
        logps = completion_log_probabilities(policy, batch)
    if was_training:
        policy.train()
    return logps

In [ ]:
trainable_parameters = [parameter for parameter in policy.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=CFG.learning_rate)


def train_one_dpo_round(pair_frame, round_seed):
    policy.train()
    rng = np.random.default_rng(round_seed)
    losses = []

    for epoch in range(CFG.dpo_epochs_per_round):
        order = rng.permutation(len(pair_frame))
        for start in range(0, len(order), CFG.train_batch_size):
            indices = order[start : start + CFG.train_batch_size]
            batch_pairs = pair_frame.iloc[indices]
            chosen = batch_pairs["chosen"].tolist()
            rejected = batch_pairs["rejected"].tolist()
            all_sequences = chosen + rejected

            # The reference probabilities do not require a gradient.
            ref_logps = reference_log_probabilities(all_sequences)

            policy.train()
            policy_batch = encode_completions(all_sequences)
            policy_logps = completion_log_probabilities(policy, policy_batch)

            split = len(chosen)
            policy_chosen, policy_rejected = policy_logps[:split], policy_logps[split:]
            ref_chosen, ref_rejected = ref_logps[:split], ref_logps[split:]

            chosen_log_ratio = policy_chosen - ref_chosen
            rejected_log_ratio = policy_rejected - ref_rejected
            logits = CFG.beta * (chosen_log_ratio - rejected_log_ratio)
            loss = -F.logsigmoid(logits).mean()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_parameters, CFG.max_grad_norm)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

            del policy_batch, policy_logps, ref_logps, loss

    policy.eval()
    return float(np.mean(losses))

## **8. Run the iterative preference loop**

The batch scored in round $t$ is sampled from the policy produced by round $t-1$. This repeated interaction is what makes data collection online/iterative.

> If every sequence receives the same length, the oracle cannot construct preferences. Increase sampling diversity or use the target-length/pLDDT objective.

In [ ]:
if CFG.reward_mode in {"plddt", "length_plddt"} and CFG.samples_per_round > 8:
    print(
        "Warning: pLDDT scoring is expensive. Consider samples_per_round <= 8, "
        "rounds <= 2 and max_new_tokens <= 96."
    )

round_frames = []
round_summaries = []

for round_index in range(CFG.rounds):
    started = time.time()
    sampled = sample_sequences(
        policy,
        n_sequences=CFG.samples_per_round,
        seed=CFG.seed + round_index,
        use_reference=False,
    )
    scored = describe_sequences(sampled)
    scored["reward"] = score_sequences(sampled)
    scored["round"] = round_index
    pairs = make_preference_pairs(scored, CFG.pairs_per_round)
    mean_loss = train_one_dpo_round(pairs, CFG.seed + round_index)

    round_frames.append(scored)
    round_summaries.append({
        "round": round_index,
        "mean_reward": scored["reward"].mean(),
        "median_reward": scored["reward"].median(),
        "mean_length": scored["length"].mean(),
        "unique_fraction": scored["sequence"].nunique() / len(scored),
        "cap_fraction": (scored["length"] >= CFG.max_new_tokens).mean(),
        "mean_dpo_loss": mean_loss,
        "minutes": (time.time() - started) / 60,
    })
    print(
        f"Round {round_index}: reward={scored['reward'].mean():.2f}, "
        f"length={scored['length'].mean():.1f}, DPO loss={mean_loss:.3f}"
    )

history = pd.concat(round_frames, ignore_index=True)
round_summary = pd.DataFrame(round_summaries)
display(round_summary.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
sns.boxplot(data=history, x="round", y="reward", color="#F6C85F", ax=axes[0])
axes[0].set_title("Oracle reward before each update")
sns.boxplot(data=history, x="round", y="length", color="#6F4EAA", ax=axes[1])
axes[1].axhline(CFG.max_new_tokens, ls="--", color="black", lw=1)
axes[1].set_title("Sequence length")
sns.lineplot(data=round_summary, x="round", y="mean_dpo_loss", marker="o", ax=axes[2])
axes[2].set_title("Mean DPO loss")
plt.tight_layout()
plt.show()

## **9. Compare the reference and aligned policies on fresh samples**

Training-batch reward can be misleading. We now draw fresh samples with a new seed from both the fixed reference and aligned policies, then compare reward, diversity and simple low-complexity warnings.

For length optimisation, pay particular attention to `cap_fraction`: a rise towards 1 means the policy has learned to avoid the end token until it reaches our artificial generation limit. This is successful optimisation of a poorly specified reward—not evidence of better proteins.

In [ ]:
evaluation_seed = CFG.seed + 10_000
reference_sequences = sample_sequences(
    policy,
    CFG.evaluation_samples,
    seed=evaluation_seed,
    use_reference=True,
)
aligned_sequences = sample_sequences(
    policy,
    CFG.evaluation_samples,
    seed=evaluation_seed,
    use_reference=False,
)


def evaluate_policy_samples(label, sequences):
    frame = describe_sequences(sequences)
    frame["reward"] = score_sequences(sequences)
    frame["policy"] = label
    return frame


evaluation = pd.concat(
    [
        evaluate_policy_samples("Reference", reference_sequences),
        evaluate_policy_samples("Aligned", aligned_sequences),
    ],
    ignore_index=True,
)

evaluation_summary = (
    evaluation.groupby("policy")
    .agg(
        mean_reward=("reward", "mean"),
        median_reward=("reward", "median"),
        mean_length=("length", "mean"),
        unique_fraction=("sequence", lambda x: x.nunique() / len(x)),
        valid_fraction=("valid", "mean"),
        mean_entropy_bits=("entropy_bits", "mean"),
        high_repetition_fraction=("max_residue_fraction", lambda x: (x > 0.35).mean()),
        cap_fraction=("length", lambda x: (x >= CFG.max_new_tokens).mean()),
    )
    .reset_index()
)
display(evaluation_summary.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
sns.boxplot(data=evaluation, x="policy", y="reward", hue="policy", legend=False, ax=axes[0])
axes[0].set_title("Fresh-sample reward")
sns.boxplot(data=evaluation, x="policy", y="length", hue="policy", legend=False, ax=axes[1])
axes[1].axhline(CFG.max_new_tokens, ls="--", color="black", lw=1)
axes[1].set_title("Fresh-sample length")
sns.boxplot(
    data=evaluation,
    x="policy",
    y="max_residue_fraction",
    hue="policy",
    legend=False,
    ax=axes[2],
)
axes[2].axhline(0.35, ls="--", color="black", lw=1)
axes[2].set_title("Largest single-residue fraction")
plt.tight_layout()
plt.show()

display(
    evaluation.sort_values("reward", ascending=False)[
        ["policy", "reward", "length", "entropy_bits", "max_residue_fraction", "sequence"]
    ].head(10)
)

### **9.1 Interpreting the result**

A convincing classroom run should show a higher **held-out** reward for the aligned policy. That alone is not enough. Ask:

- Did diversity collapse?
- Did sequences become repetitive or compositionally extreme?
- Did length merely hit the decoding cap?
- Would an independent predictor agree with the training oracle?
- Is the result still good under a different random seed?

For a better length specification, switch to `reward_mode="target_length"`. For a multi-objective design, combine rank-normalised metrics and inspect the individual components as well as the total reward.

## **10. Optional CLEAN extension: optimise an EC prediction**

**CLEAN** (Contrastive Learning enabled Enzyme ANnotation) predicts Enzyme Commission (EC) classes from sequence representations. Its max-separation output lists predictions as `EC:number/distance`; for a selected EC class, a smaller distance is preferable.

The official local workflow currently requires a large pretrained-data bundle and ESM-1b weights. Installing it inside every teaching run would dominate the practical, so the cells below provide the two reusable pieces needed to connect CLEAN:

1. export the sequences from a DPO round to FASTA;
2. parse CLEAN's returned max-separation CSV into a target-EC reward.

For a truly automated online loop, run CLEAN once as a persistent service and replace `score_sequences` with a callback to that service. Reinstalling/reloading CLEAN every round is not a sensible online workflow.

The example parser assigns a deliberately poor fallback reward when the target EC is absent from CLEAN's reported hits. That penalty is a modelling decision—not a calibrated probability—and should be examined for each target.

In [ ]:
def export_fasta(sequences, path="clean_candidates.fasta"):
    path = Path(path)
    with path.open("w") as handle:
        for index, sequence in enumerate(sequences):
            handle.write(f">seq_{index:04d}\n{sequence}\n")
    print(f"Wrote {len(sequences)} sequences to {path.resolve()}")
    return path


def parse_clean_maxsep(path, target_ec, missing_penalty=-20.0):
    # Parse official CLEAN max-separation output into rewards (higher is better).
    target_ec = target_ec.removeprefix("EC:")
    rewards = {}

    with Path(path).open() as handle:
        for raw_line in handle:
            fields = [field.strip() for field in raw_line.strip().split(",") if field.strip()]
            if not fields:
                continue
            sequence_id, predictions = fields[0], fields[1:]
            target_distance = None
            for prediction in predictions:
                match = re.fullmatch(r"EC:([^/]+)/([0-9.eE+-]+)", prediction)
                if match and match.group(1) == target_ec:
                    target_distance = float(match.group(2))
                    break
            rewards[sequence_id] = (
                -target_distance if target_distance is not None else float(missing_penalty)
            )

    return pd.Series(rewards, name=f"CLEAN reward for EC:{target_ec}")


# Example hand-off after a generation round:
# fasta_path = export_fasta(history.query("round == @history['round'].max()")['sequence'])
# Run CLEAN_infer_fasta.py with the exported FASTA, then upload its *_maxsep.csv.
# clean_rewards = parse_clean_maxsep("candidates_maxsep.csv", target_ec="1.1.1.1")

### **10.1 Combining length, pLDDT and CLEAN**

Raw length, pLDDT and CLEAN distance have different units and ranges. Combining the raw values would make the largest numerical scale dominate. A transparent classroom approach is to convert each metric to a percentile rank within the current batch and then apply explicit weights.

In [ ]:
def composite_reward(metric_table, weights):
    # Weighted sum of within-batch percentile ranks; higher must be better for every column.
    unknown = set(weights) - set(metric_table.columns)
    if unknown:
        raise KeyError(f"Missing metric columns: {sorted(unknown)}")
    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError("Composite weights must sum to 1.")

    total = np.zeros(len(metric_table), dtype=float)
    for metric, weight in weights.items():
        total += weight * rank_to_unit_interval(metric_table[metric].to_numpy())
    return total


# Once all three oracle columns exist for the same generated batch:
# metrics = pd.DataFrame({
#     "length": length_scores(sequences),
#     "plddt": mean_plddt_scores(sequences),
#     "clean": clean_rewards_in_sequence_order,
# })
# metrics["composite_reward"] = composite_reward(
#     metrics,
#     weights={"length": 0.20, "plddt": 0.40, "clean": 0.40},
# )

## **11. Save the lightweight adapter (optional)**

The adapter contains only the learned LoRA parameters, not a copy of the full base checkpoint. Saving locally or downloading from Colab does **not** modify the original model or any GitHub repository.

In [ ]:
SAVE_ADAPTER = False

if SAVE_ADAPTER:
    adapter_directory = Path("ProtGPT3-112M-online-DPO-adapter")
    policy.save_pretrained(adapter_directory)
    tokenizer.save_pretrained(adapter_directory)
    print(f"Saved adapter to {adapter_directory.resolve()}")
else:
    print("Adapter not saved. Set SAVE_ADAPTER=True if you want to keep this run.")

## **12. Suggested exercises**

1. **Expose reward hacking.** Run the default length reward and measure how often sequences reach the generation cap.
2. **Repair the objective.** Change to `target_length` and compare the result with unbounded length maximisation.
3. **Change the oracle.** Run one small pLDDT round. Does mean pLDDT increase on fresh samples? What happens to diversity?
4. **Use multiple objectives.** Compare a weighted sum with a Pareto-front selection rule. Which trade-offs become hidden by a single scalar?
5. **Test generalisation.** Evaluate with a second predictor that was not used for training, then propose a minimal experimental validation plan.

### **Discussion questions**

- When does an oracle become part of the model's training distribution rather than an independent evaluator?
- Why can a model achieve a high pLDDT reward without producing the desired function?
- What would change if preference pairs came from experiments rather than predictions?
- Which safeguards would you add before optimising a pathogenicity-, toxicity- or host-interaction-related score?

## **References and resources**

- Stocco, F., Garibbo, M. & Ferruz, N. **Steering Generative Models for Protein Design: Aligning and Conditioning Strategies.** arXiv:2511.21476, v2 (2026). https://arxiv.org/abs/2511.21476
- AI4PDLab. **ProtRL: Reinforcement Learning for Protein Language Models.** https://github.com/AI4PDLab/ProtRL
- AI4PD. **ProtGPT3-112M model card.** https://huggingface.co/AI4PD/ProtGPT3-112M
- Rafailov, R. *et al.* **Direct Preference Optimization: Your Language Model is Secretly a Reward Model.** NeurIPS (2023). https://arxiv.org/abs/2305.18290
- Lin, Z. *et al.* **Evolutionary-scale prediction of atomic-level protein structure with a language model.** *Science* (2023). https://doi.org/10.1126/science.ade2574
- Yu, T. *et al.* **Enzyme function prediction using contrastive learning.** *Science* (2023). https://doi.org/10.1126/science.adf2465

---

**Reproducibility note:** generative sampling and GPU kernels can remain partly stochastic despite fixed seeds. Record package versions, configuration, random seed, generated sequences and oracle outputs for any result used beyond the classroom.